# Fusion、donation 与存储复用

对应 R02/R05/R06。CPU 执行带 **VERSION-SKEW**；静态 accounted bytes 和执行后可见 payload 都不是实测设备峰值。
本 Notebook 复查 11 组捕获，并重新执行 owned/external-view donation 对照。详见 [fusion-and-memory.md](fusion-and-memory.md)。

In [1]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "upstream-sources.lock").is_file())
sys.path.insert(0, str(root / "research/software-stack/tools"))
from verify_fusion_memory import verify
result = verify(root / "artifacts/jax-stack/fusion-memory-002")
cases = {c["case"]: c for c in result["cases"]}
print("verified", len(cases), "cases and", result["artifact_count"], "artifacts")
for name, c in cases.items():
    m = c["memory_analysis"]
    print(name, {"alias":m["alias_size_in_bytes"], "temp":m["temp_size_in_bytes"], "static_accounted":m["accounted_bytes"], "visible_payload":c["visible_distinct_payload_bytes_after_call"]})

verified 11 cases and 2028 artifacts
chain-fused {'alias': 0, 'temp': 0, 'static_accounted': 8192, 'visible_payload': 8192}
chain-donated {'alias': 4096, 'temp': 0, 'static_accounted': 4096, 'visible_payload': 4096}
chain-external-view {'alias': 4096, 'temp': 0, 'static_accounted': 4096, 'visible_payload': 8192}
chain-no-instruction-fusion {'alias': 0, 'temp': 0, 'static_accounted': 8192, 'visible_payload': 8192}
chain-no-fusion-no-wrapper {'alias': 0, 'temp': 0, 'static_accounted': 8192, 'visible_payload': 8192}
reshape-not-donated {'alias': 0, 'temp': 0, 'static_accounted': 8192, 'visible_payload': 8192}
reshape-donated {'alias': 4096, 'temp': 4096, 'static_accounted': 8192, 'visible_payload': 4096}
reduce-donation-unused {'alias': 0, 'temp': 4224, 'static_accounted': 8324, 'visible_payload': 4100}
identity {'alias': 0, 'temp': 0, 'static_accounted': 8192, 'visible_payload': 8192}
matmul-bias {'alias': 0, 'temp': 0, 'static_accounted': 512, 'visible_payload': 512}
matmul-bias-no-fusi

## 相同存储占用不代表相同中间读写

禁用 `fusion` 后仍有三个单算子 wrapper；同时禁用 `fusion-wrapper` 才在这条链中留下独立 sin/cos/tanh。
两者仍能按存活区间复用一个输出 allocation，temp bytes 为零。

In [2]:
for name in ["chain-fused", "chain-no-instruction-fusion", "chain-no-fusion-no-wrapper"]:
    c = cases[name]
    print(name, [op["opcode"] for op in c["graph"]["entry_operations"]], "model bytes:", c["cost_analysis"]["bytes accessed"])
print(cases["chain-no-fusion-no-wrapper"]["storage_analysis"]["shared_storage_pairs"])

chain-fused ['kParameter', 'kFusion'] model bytes: 8192.0
chain-no-instruction-fusion ['kParameter', 'kFusion', 'kFusion', 'kFusion'] model bytes: 24576.0
chain-no-fusion-no-wrapper ['kParameter', 'kSin', 'kCos', 'kTanh'] model bytes: 24576.0
[{'allocation': 0, 'values': ['sin.1{}', 'cos.1{}'], 'overlap_bytes': 4096, 'live_ranges': [(1, 2), (2, 3)], 'boundary_touch': True}, {'allocation': 0, 'values': ['sin.1{}', 'tanh.1{}'], 'overlap_bytes': 4096, 'live_ranges': [(1, 2), (3, 4)], 'boundary_touch': False}, {'allocation': 0, 'values': ['cos.1{}', 'tanh.1{}'], 'overlap_bytes': 4096, 'live_ranges': [(2, 3), (3, 4)], 'boundary_touch': True}]


## 重新执行 donation / 外部 view 对照

编译器 alias 计划与运行时能否取得独占存储是不同层次。这里刻意保留 NumPy view 进行对照。
不要从本次 input 仍有效推导跨版本 API 保证；调用方应遵守 donation 后不再使用输入的合同。

In [3]:
import jax, jax.numpy as jnp, numpy as np
make = jax.jit(lambda: jnp.arange(1024,dtype=jnp.float32)/np.float32(2048)-np.float32(.25))
function = jax.jit(lambda x:jnp.tanh(jnp.cos(jnp.sin(x))), donate_argnums=(0,))
compiled = function.lower(jax.ShapeDtypeStruct((1024,),jnp.float32)).compile()
reference_input = np.arange(1024,dtype=np.float32)/np.float32(2048)-np.float32(.25)
reference = np.tanh(np.cos(np.sin(reference_input.astype(np.float64))))
for external in [False, True]:
    x = make(); x.block_until_ready()
    old_pointer = x.unsafe_buffer_pointer()
    view = np.asarray(x) if external else None
    y = compiled(x); y.block_until_ready()
    same = old_pointer == y.unsafe_buffer_pointer()
    deleted = x.is_deleted()
    np.testing.assert_allclose(np.asarray(y), reference, rtol=2e-5, atol=2e-5)
    if external:
        np.testing.assert_array_equal(view, reference_input)
        assert not same
    else:
        assert same and deleted
    print({"external_view":external, "input_deleted":deleted, "pointer_reused":same})

{'external_view': False, 'input_deleted': True, 'pointer_reused': True}
{'external_view': True, 'input_deleted': False, 'pointer_reused': False}


I0914 23:26:52.262946  371767 pjrt_client.cc:579] PjRt-IFRT device count: total=1, addressable=1
I0914 23:26:52.262967  371767 pjrt_client.cc:583] Addressable PjRt-IFRT device: CpuDevice(id=0)


## 重算逻辑区间，核对诊断标签

下面按 dump 的闭区间重算。逻辑值可共享物理存储，这条曲线仍不能称作实际内存峰值。

In [4]:
storage = cases["reduce-donation-unused"]["storage_analysis"]
print("inclusive logical bytes:", storage["inclusive_logical_value_bytes_by_time"])
print("recomputed logical peak:", storage["inclusive_logical_peak_first_time"], storage["inclusive_logical_peak_bytes"])
print("dump labelled peak:", storage["live_range"]["reported_peak_time"], storage["live_range"]["reported_peak_value_bytes"])
assert not storage["reported_peak_is_inclusive_logical_max"]
for name in ["reshape-not-donated", "reshape-donated"]:
    print(name, cases[name]["memory_analysis"])

inclusive logical bytes: [4096, 4100, 8196, 8324, 4232, 4100]
recomputed logical peak: 3 8324
dump labelled peak: 4 4232
reshape-not-donated {'argument_size_in_bytes': 4096, 'output_size_in_bytes': 4096, 'alias_size_in_bytes': 0, 'temp_size_in_bytes': 0, 'generated_code_size_in_bytes': 0, 'accounted_bytes': 8192}
reshape-donated {'argument_size_in_bytes': 4096, 'output_size_in_bytes': 4096, 'alias_size_in_bytes': 4096, 'temp_size_in_bytes': 4096, 'generated_code_size_in_bytes': 0, 'accounted_bytes': 8192}
